In [19]:
from ultralytics import YOLO
import cv2

from sort.sort import Sort

import numpy as np
import pandas as pd

In [20]:
log_df = pd.DataFrame(columns=["id", "x1", "y1", "x2", "y2", "class", "confidence"])

In [21]:
model = YOLO("ultra_final_final.pt")

In [22]:
tracker = Sort(max_age=30, min_hits=3, iou_threshold=0.2)

In [23]:
cap = cv2.VideoCapture(r"D:\sst\20250920_133619_1.mp4")

ret, frame = cap.read()

print(frame.shape)
xbuffer = 10
ybuffer = 8
roi = [xbuffer, ybuffer, frame.shape[1]-xbuffer, frame.shape[0]-ybuffer]


(3840, 2160, 3)


In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLOv8 inference
    results = model.predict(source=frame, show=False, conf=0.3)


    # Get first result (single frame)
    result = results[0]

    detections = np.empty((0, 5))

    # Extract bounding boxes
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
        # x1, y1, x2, y2 = [int(v) for v in box.xyxy[0]]


        conf = box.conf[0]           
        cls = int(box.cls[0])

        CurrentArray = np.array([x1, y1, x2, y2, conf])
        detections = np.vstack((detections, CurrentArray))

    TrackResults = tracker.update(detections)
    for tracking in TrackResults:
        x1, y1, x2, y2, id = tracking
        x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

        cx, cy = (x2-x1)/2, (y2-y1)/2
        # cx, cy = (x1 + x2) // 2, (y1 + y2) // 2


        if roi[0]< cx < roi[2] and roi[1] < cy < roi[3] and int(id) not in log_df['id'].values:
            log_df.loc[len(log_df)] = [id, x1, y1, x2, y2, model.names[cls], float(conf)]

        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 3)
        cv2.putText(frame, f'{id}', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
        # print(track)
    
    # Display result
    cv2.imshow("YOLOv8 Webcam with Boxes", frame)

    # Exit on ' '
    if cv2.waitKey(1) & 0xFF == ord(' '):
        break

cap.release()
cv2.destroyAllWindows()


0: 1024x576 (no detections), 78.2ms
Speed: 7.4ms preprocess, 78.2ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 103.4ms
Speed: 9.1ms preprocess, 103.4ms inference, 0.8ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 103.3ms
Speed: 9.6ms preprocess, 103.3ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 92.8ms
Speed: 8.1ms preprocess, 92.8ms inference, 0.8ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 80.7ms
Speed: 8.2ms preprocess, 80.7ms inference, 0.8ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 81.5ms
Speed: 8.2ms preprocess, 81.5ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 88.4ms
Speed: 8.2ms preprocess, 88.4ms inference, 0.9ms postprocess per image at shape (1, 3, 1024, 576)

0: 1024x576 (no detections), 110.0ms
Speed: 8.6ms 

In [25]:
print(model.names)

{0: 'target'}


In [26]:
print(log_df)

       id    x1    y1    x2    y2   class  confidence
0   474.0    25  1139  2137  2703  target    0.416118
1   475.0   -43    -6  2204  2523  target    0.411442
2   480.0  1426  2615  1532  2719  target    0.415879
3   481.0   768  1587  1154  1975  target    0.933592
4   488.0  1125  3656  1784  3836  target    0.382571
5   489.0  1191  1737  1313  1855  target    0.524201
6   490.0  1229  3556  1832  3841  target    0.489943
7   493.0  1185  2431  1255  2495  target    0.302384
8   494.0   886  1643  1354  2132  target    0.842859
9   501.0  1977     3  2160   782  target    0.442569
10  503.0  1994  1842  2155  2785  target    0.448602
11  502.0  1979  1481  2158  2794  target    0.448602
12  514.0     5  2389   283  3599  target    0.806900
13  518.0  1440  2652  1548  2754  target    0.590320
14  521.0   493  1429   905  1851  target    0.820016
15  525.0    34  1205  2132  3872  target    0.367057
16  526.0    64   876  1293  2204  target    0.764148


In [27]:
log_df.to_csv("output.csv", index=False)